# Sleep Deprivation Classification 2

This notebook keeps the preprocessing contract from `raw_dataset_creation.ipynb` and the subject-wise split logic from `Sleep_Deprivation_Classification_1.ipynb`.
It is now focused on task 1 from the project proposal, which is the main `NS` versus `SD` classification task.
The notebook compares only the strongest EEG CNN models from the earlier experiments.
It also follows the file 1 workflow more closely by using train and validation for model comparison first.
The held out test split is kept for a separate final evaluation step.

Models included in the same training pipeline:
- `residual_eeg_cnn`: deeper EEG specific residual CNN with batch normalization, layer normalization, dropout, and squeeze excitation
- `two_branch_eeg_cnn`: EEG CNN with one temporal branch and one channel branch before fusion
- `residual_eeg_cnn_cv5`: optional subject-wise cross-validation for the strongest model family

Metrics used during the main comparison:
- Validation accuracy
- Validation F1 score
- Validation balanced accuracy
- Validation AUROC
- Validation session level F1 score
- Validation session level balanced accuracy

Metrics reported in the optional final test step:
- Test accuracy
- Test F1 score
- Test balanced accuracy
- Test AUROC
- Test session level F1 score
- Full `classification_report` from scikit-learn
- Confusion matrix


In [30]:
# This cell imports the libraries used in the notebook.
# We keep the imports in one place so the rest of the notebook stays clean.

import copy
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm


In [31]:
# This cell makes the notebook reproducible.
# Using the same seed gives you the same data split and nearly the same training path.

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


In [32]:
# This cell mounts Google Drive when you run the notebook in Colab.
# If you are not in Colab, this cell just prints a message and moves on.

IS_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IS_COLAB = True
    print("Google Drive mounted.")
except Exception:
    print("Not running in Colab, skipping Google Drive mount.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.


In [33]:
# This cell sets the main project paths.
# If MANUAL_PROJECT_ROOT is set, the notebook uses that path directly.
# If MANUAL_PROJECT_ROOT is None, the notebook tries to find the project root automatically.

def find_project_root(start=None):
    if start is None:
        start = Path.cwd().resolve()
    start = start.resolve()

    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "preprocessed_data").exists() and (candidate / "code").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root that contains data/preprocessed_data and code/.")


# This is the Colab project path provided for this notebook.
# In Colab it is used automatically.
# Outside Colab the notebook falls back to auto-detection.
MANUAL_PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/G/CSCI 5527/Project" if IS_COLAB else None

if MANUAL_PROJECT_ROOT is not None:
    PROJECT_ROOT = Path(MANUAL_PROJECT_ROOT)
else:
    PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data" / "preprocessed_data"
SPLIT_DIR = PROJECT_ROOT / "code" / "sleep_deprivation_classification" / "splits_cls2"
EXPERIMENT_ROOT = PROJECT_ROOT / "code" / "sleep_deprivation_classification" / "experiments_cls2"
RAW_NB_PATH = PROJECT_ROOT / "code" / "data_preprocessing" / "raw_dataset_creation.ipynb"
RAW_PY_PATH = PROJECT_ROOT / "code" / "data_preprocessing" / "raw_dataset_creation.py"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("SPLIT_DIR    :", SPLIT_DIR)
print("EXPERIMENTS  :", EXPERIMENT_ROOT)


PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/G/CSCI 5527/Project
DATA_DIR     : /content/drive/MyDrive/Colab Notebooks/G/CSCI 5527/Project/data/preprocessed_data
SPLIT_DIR    : /content/drive/MyDrive/Colab Notebooks/G/CSCI 5527/Project/code/sleep_deprivation_classification/splits_cls2
EXPERIMENTS  : /content/drive/MyDrive/Colab Notebooks/G/CSCI 5527/Project/code/sleep_deprivation_classification/experiments_cls2


## Preprocessing Summary

The exported Python file below is created directly from `raw_dataset_creation.ipynb`.

The dataset preparation pipeline is:
- Read EEGLAB `.set` files for the `eyesopen` task.
- Keep EEG channels only.
- Re-reference to average reference.
- Bandpass filter from `1 Hz` to `40 Hz`.
- Resample from `500 Hz` to `250 Hz`.
- Build non-overlapping `5 second` epochs, which gives width `1250` samples.
- Clip amplitudes at `100 microvolts` and reject epochs with too many saturated channels.
- Z-score each epoch channel-wise.
- Save raw EEG epochs to `X_eeg.npy` with shape `(N, 61, 1250)`.
- Convert each epoch to a repeated 3-channel image tensor and save to `X_images.npy` with shape `(N, 3, 61, 1250)`.
- Save labels to `y_labels.npy` where `0 = NS` and `1 = SD`.
- Save subject IDs to `groups.npy` for subject-wise splitting.


In [34]:
# This cell sets the main project paths.
# For Colab, set MANUAL_PROJECT_ROOT to your project folder.

from pathlib import Path

def find_project_root(start=None):
    if start is None:
        start = Path.cwd().resolve()
    start = start.resolve()

    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "preprocessed_data").exists() and (candidate / "code").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root that contains data/preprocessed_data and code/.")


MANUAL_PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/CSCI 5527/Project"

if MANUAL_PROJECT_ROOT is not None:
    PROJECT_ROOT = Path(MANUAL_PROJECT_ROOT)
else:
    PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data" / "preprocessed_data"
SPLIT_DIR = PROJECT_ROOT / "code" / "sleep_deprivation_classification" / "splits_cls2"
EXPERIMENT_ROOT = PROJECT_ROOT / "code" / "sleep_deprivation_classification" / "experiments_cls2"
RAW_NB_PATH = PROJECT_ROOT / "code" / "data_preprocessing" / "raw_dataset_creation.ipynb"
RAW_PY_PATH = PROJECT_ROOT / "code" / "data_preprocessing" / "raw_dataset_creation.py"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("SPLIT_DIR    :", SPLIT_DIR)
print("EXPERIMENTS  :", EXPERIMENT_ROOT)


PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/CSCI 5527/Project
DATA_DIR     : /content/drive/MyDrive/Colab Notebooks/CSCI 5527/Project/data/preprocessed_data
SPLIT_DIR    : /content/drive/MyDrive/Colab Notebooks/CSCI 5527/Project/code/sleep_deprivation_classification/splits_cls2
EXPERIMENTS  : /content/drive/MyDrive/Colab Notebooks/CSCI 5527/Project/code/sleep_deprivation_classification/experiments_cls2


In [35]:
metadata_path = DATA_DIR / "eeg_metadata.csv"
labels_path = DATA_DIR / "y_labels.npy"
groups_path = DATA_DIR / "groups.npy"
x_eeg_path = DATA_DIR / "X_eeg.npy"
x_images_path = DATA_DIR / "X_images.npy"

metadata = pd.read_csv(metadata_path)
y_all = np.load(labels_path, allow_pickle=True)
groups_all = np.load(groups_path, allow_pickle=True)
X_eeg = np.load(x_eeg_path, mmap_mode="r")
X_images = np.load(x_images_path, mmap_mode="r")

print("metadata shape:", metadata.shape)
print("y shape       :", y_all.shape)
print("groups shape  :", groups_all.shape)
print("X_eeg shape   :", X_eeg.shape, X_eeg.dtype)
print("X_images shape:", X_images.shape, X_images.dtype)


metadata shape: (8300, 13)
y shape       : (8300,)
groups shape  : (8300,)
X_eeg shape   : (8300, 61, 1250) float32
X_images shape: (8300, 3, 61, 1250) uint8


In [36]:
def verify_alignment(metadata, y, groups, x_eeg, x_images):
    # This check makes sure every file points to the same samples in the same order.
    n_meta = len(metadata)
    n_y = len(y)
    n_groups = len(groups)
    n_eeg = len(x_eeg)
    n_images = len(x_images)

    if not (n_meta == n_y == n_groups == n_eeg == n_images):
        raise ValueError(
            f"Length mismatch: metadata={n_meta}, y={n_y}, groups={n_groups}, X_eeg={n_eeg}, X_images={n_images}"
        )

    if not np.array_equal(metadata["label"].to_numpy(), y):
        raise ValueError("Mismatch between metadata['label'] and y_labels.npy")

    if not np.array_equal(metadata["subject_id"].astype(str).to_numpy(), groups.astype(str)):
        raise ValueError("Mismatch between metadata['subject_id'] and groups.npy")

    print("Alignment check passed.")


verify_alignment(metadata, y_all, groups_all, X_eeg, X_images)


Alignment check passed.


In [37]:
# These are the basic experiment settings.
# By default the notebook uses 6000 samples so you can try many models in one run.

USE_SUBSET = False
MAX_TOTAL_SAMPLES = 8300
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
NUM_CV_FOLDS = 5

assert abs(TRAIN_SIZE + VAL_SIZE + TEST_SIZE - 1.0) < 1e-8


In [38]:
def build_subject_subset_indices(metadata, max_total_samples=None):
    # This selects the active working subset.
    # It adds subjects until the sample budget is reached.
    all_indices = np.arange(len(metadata))
    if max_total_samples is None or max_total_samples >= len(all_indices):
        return all_indices

    subject_counts = metadata.groupby("subject_id").size().sort_values()
    chosen_subjects = []
    running_total = 0

    for subject_id, count in subject_counts.items():
        if running_total >= max_total_samples:
            break
        chosen_subjects.append(subject_id)
        running_total += int(count)

    subset_mask = metadata["subject_id"].isin(chosen_subjects).to_numpy()
    subset_indices = all_indices[subset_mask]

    if len(subset_indices) > max_total_samples:
        subset_indices = subset_indices[:max_total_samples]

    return np.sort(subset_indices)


active_indices = build_subject_subset_indices(
    metadata=metadata,
    max_total_samples=MAX_TOTAL_SAMPLES if USE_SUBSET else None,
)

active_metadata = metadata.iloc[active_indices].reset_index(drop=True)
active_y = y_all[active_indices]
active_groups = groups_all[active_indices]

print("Using samples :", len(active_indices))
print("Using subjects:", active_metadata["subject_id"].nunique())
print("Label counts  :", pd.Series(active_y).value_counts().sort_index().to_dict())
print("EEG example   :", X_eeg[active_indices[0]].shape, "-> will become", (1, *X_eeg[active_indices[0]].shape))
print("Image example :", X_images[active_indices[0]].shape)


Using samples : 8300
Using subjects: 71
Label counts  : {0: 4249, 1: 4051}
EEG example   : (61, 1250) -> will become (1, 61, 1250)
Image example : (3, 61, 1250)


In [39]:
def make_group_splits(active_indices, y, groups, train_size=0.70, val_size=0.15, test_size=0.15, seed=42):
    # This split is subject-wise.
    # A subject can only appear in train, validation, or test, never in more than one split.
    assert abs(train_size + val_size + test_size - 1.0) < 1e-8

    subset_positions = np.arange(len(active_indices))

    gss_1 = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=seed)
    train_pos, temp_pos = next(gss_1.split(subset_positions, y, groups))

    relative_val = val_size / (val_size + test_size)
    temp_y = y[temp_pos]
    temp_groups = groups[temp_pos]

    gss_2 = GroupShuffleSplit(n_splits=1, train_size=relative_val, random_state=seed + 1)
    val_pos_local, test_pos_local = next(gss_2.split(np.arange(len(temp_pos)), temp_y, temp_groups))

    val_pos = temp_pos[val_pos_local]
    test_pos = temp_pos[test_pos_local]

    splits = {
        "train": np.sort(active_indices[train_pos]),
        "val": np.sort(active_indices[val_pos]),
        "test": np.sort(active_indices[test_pos]),
    }
    return splits


splits = make_group_splits(
    active_indices=active_indices,
    y=active_y,
    groups=active_groups,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
    seed=SEED,
)


In [40]:
def assert_no_group_overlap(groups, splits):
    # This makes the leakage check explicit.
    train_groups = set(groups[splits["train"]].astype(str))
    val_groups = set(groups[splits["val"]].astype(str))
    test_groups = set(groups[splits["test"]].astype(str))

    assert len(train_groups & val_groups) == 0, "Subject overlap between train and val"
    assert len(train_groups & test_groups) == 0, "Subject overlap between train and test"
    assert len(val_groups & test_groups) == 0, "Subject overlap between val and test"
    print("No subject leakage across splits.")


def summarize_split(name, indices, metadata, y):
    # This prints a simple summary of each split.
    split_meta = metadata.iloc[indices].copy()
    print(f"\n===== {name.upper()} SPLIT =====")
    print("num samples   :", len(indices))
    print("num subjects  :", split_meta["subject_id"].nunique())
    print("num recordings:", split_meta["file_path"].nunique())
    print("label counts  :", pd.Series(y[indices]).value_counts().sort_index().to_dict())
    print("session counts:", split_meta["session"].value_counts().to_dict())


assert_no_group_overlap(groups_all, splits)
for split_name, split_indices in splits.items():
    summarize_split(split_name, split_indices, metadata, y_all)


No subject leakage across splits.

===== TRAIN SPLIT =====
num samples   : 5738
num subjects  : 49
num recordings: 96
label counts  : {0: 2940, 1: 2798}
session counts: {'ses-1': 2940, 'ses-2': 2798}

===== VAL SPLIT =====
num samples   : 1246
num subjects  : 11
num recordings: 21
label counts  : {0: 649, 1: 597}
session counts: {'ses-1': 649, 'ses-2': 597}

===== TEST SPLIT =====
num samples   : 1316
num subjects  : 11
num recordings: 22
label counts  : {0: 660, 1: 656}
session counts: {'ses-1': 660, 'ses-2': 656}


In [41]:
for split_name, split_indices in splits.items():
    np.save(SPLIT_DIR / f"{split_name}_indices.npy", split_indices)
    split_df = metadata.iloc[split_indices].copy()
    split_df["global_index"] = split_indices
    split_df.to_csv(SPLIT_DIR / f"{split_name}_metadata.csv", index=False)

summary = {
    "seed": SEED,
    "use_subset": USE_SUBSET,
    "max_total_samples": int(MAX_TOTAL_SAMPLES),
    "train_size": TRAIN_SIZE,
    "val_size": VAL_SIZE,
    "test_size": TEST_SIZE,
    "splits": {},
}

for split_name, split_indices in splits.items():
    split_meta = metadata.iloc[split_indices]
    summary["splits"][split_name] = {
        "num_samples": int(len(split_indices)),
        "num_subjects": int(split_meta["subject_id"].nunique()),
        "label_counts": {str(k): int(v) for k, v in pd.Series(y_all[split_indices]).value_counts().sort_index().to_dict().items()},
        "session_counts": {str(k): int(v) for k, v in split_meta["session"].value_counts().to_dict().items()},
    }

with open(SPLIT_DIR / "split_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved split artifacts to:", SPLIT_DIR)


Saved split artifacts to: /content/drive/MyDrive/Colab Notebooks/CSCI 5527/Project/code/sleep_deprivation_classification/splits_cls2


In [42]:
# This helper makes subject-wise cross-validation folds.
# Each fold keeps whole subjects together so the model never sees the same subject in train and validation.

def make_group_kfold_splits(active_indices, y, groups, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    fold_splits = []
    subset_positions = np.arange(len(active_indices))

    for fold_id, (train_pos, val_pos) in enumerate(gkf.split(subset_positions, y, groups), start=1):
        fold_splits.append({
            "fold_id": fold_id,
            "train": np.sort(active_indices[train_pos]),
            "val": np.sort(active_indices[val_pos]),
            "test": np.sort(active_indices[val_pos]),
        })
    return fold_splits


cv_splits = make_group_kfold_splits(active_indices, active_y, active_groups, n_splits=NUM_CV_FOLDS)
print("Prepared subject-wise CV folds:", len(cv_splits))


Prepared subject-wise CV folds: 5


## Why These Training Tricks Are Used

The choices below follow the lecture material and are reasonable for this dataset.
They are used to improve generalization on the main classification task without changing the metrics.

- `BatchNorm2d` is used in convolution blocks to stabilize feature scale across layers.
- `LayerNorm` is used in the classifier head of the deeper EEG CNN to stabilize the final embedding without depending on spatial dimensions.
- `ReLU`, `LeakyReLU`, and `SiLU` are used instead of sigmoid because the lecture notes emphasize avoiding saturating activations in deep models.
- `Kaiming` initialization is used for convolution layers with ReLU style activations.
- `AdamW` is used as an adaptive optimizer, which the lecture notes motivate for deep models with uneven gradient scales.
- `ReduceLROnPlateau` gives a staircase style learning-rate reduction when validation performance stalls.
- `Early stopping` is used because validation performance is the practical stopping rule highlighted in the lecture.
- `Dropout`, `weight decay`, and `label smoothing` are added for regularization.
- `Class-weighted cross-entropy` and weighted mini-batch sampling reduce class bias during training.
- Mild time jitter and time crop are used only as small EEG-specific data augmentation.
- Session-level validation is used for checkpoint selection because the real task signal is stronger at the recording-session level than at the single-epoch level.


In [43]:
# These dataset classes load either raw EEG matrices or repeated 3 channel EEG images.
# The EEG dataset can also apply light time jitter and random time crops during training.

def apply_eeg_time_augmentation(x, augment_config):
    # x has shape (1, 61, 1250)
    # We only change the time axis because the channel axis is the sensor layout.
    if augment_config is None:
        return x

    out = x
    _, _, width = out.shape

    time_jitter_max = int(augment_config.get("time_jitter_max", 0))
    if time_jitter_max > 0:
        shift = int(np.random.randint(-time_jitter_max, time_jitter_max + 1))
        out = torch.roll(out, shifts=shift, dims=2)

    crop_fraction = float(augment_config.get("time_crop_fraction", 1.0))
    if 0.0 < crop_fraction < 1.0:
        crop_width = max(32, int(width * crop_fraction))
        if crop_width < width:
            start = int(np.random.randint(0, width - crop_width + 1))
            cropped = out[:, :, start:start + crop_width].unsqueeze(0)
            out = F.interpolate(cropped, size=(out.shape[1], width), mode="bilinear", align_corners=False).squeeze(0)

    return out


class EEGMatrixDataset(Dataset):
    def __init__(self, x_eeg_path, y_path, groups_path, metadata_csv_path, indices, is_train=False, augment_config=None):
        self.x_eeg = np.load(x_eeg_path, mmap_mode="r")
        self.y = np.load(y_path, allow_pickle=True)
        self.groups = np.load(groups_path, allow_pickle=True)
        self.metadata = pd.read_csv(metadata_csv_path)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.is_train = is_train
        self.augment_config = augment_config or {}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        global_idx = int(self.indices[idx])
        x = np.array(self.x_eeg[global_idx], dtype=np.float32, copy=True)
        x = torch.from_numpy(x).unsqueeze(0)
        if self.is_train:
            x = apply_eeg_time_augmentation(x, self.augment_config)
        y = torch.tensor(int(self.y[global_idx]), dtype=torch.long)
        meta = self.metadata.iloc[global_idx].to_dict()
        meta["global_index"] = global_idx
        meta["group"] = str(self.groups[global_idx])
        return x, y, meta

    def get_labels(self):
        return self.y[self.indices]


class EEGImageDataset(Dataset):
    def __init__(self, x_images_path, y_path, groups_path, metadata_csv_path, indices, transform=None):
        self.x_images = np.load(x_images_path, mmap_mode="r")
        self.y = np.load(y_path, allow_pickle=True)
        self.groups = np.load(groups_path, allow_pickle=True)
        self.metadata = pd.read_csv(metadata_csv_path)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        global_idx = int(self.indices[idx])
        x = np.array(self.x_images[global_idx], dtype=np.float32, copy=True) / 255.0
        x = torch.from_numpy(x)
        if self.transform is not None:
            x = self.transform(x)
        y = torch.tensor(int(self.y[global_idx]), dtype=torch.long)
        meta = self.metadata.iloc[global_idx].to_dict()
        meta["global_index"] = global_idx
        meta["group"] = str(self.groups[global_idx])
        return x, y, meta

    def get_labels(self):
        return self.y[self.indices]


In [44]:
# This transform resizes the EEG image to the size expected by image backbones.
# We keep normalization values close to ImageNet when we use pretrained CNN weights.

def build_image_transform(image_size=224):
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)

    def transform(x):
        x = x.unsqueeze(0)
        x = F.interpolate(x, size=(image_size, image_size), mode="bilinear", align_corners=False)
        x = x.squeeze(0)
        return (x - mean) / std

    return transform


def build_dataloaders(model_config, split_indices=None):
    # This helper builds train, validation, and test loaders for one model config.
    # EEG models read X_eeg.npy and image models read X_images.npy.
    # We also attach the right augmentation settings for the training split.
    batch_size = model_config["batch_size"]
    num_workers = model_config.get("num_workers", 0)
    pin_memory = torch.cuda.is_available()
    split_indices = split_indices or splits
    eeg_augment_config = {
        "time_jitter_max": model_config.get("time_jitter_max", 0),
        "time_crop_fraction": model_config.get("time_crop_fraction", 1.0),
    }

    if model_config["input_type"] == "eeg":
        train_dataset = EEGMatrixDataset(
            x_eeg_path, labels_path, groups_path, metadata_path, split_indices["train"],
            is_train=True, augment_config=eeg_augment_config
        )
        val_dataset = EEGMatrixDataset(x_eeg_path, labels_path, groups_path, metadata_path, split_indices["val"], is_train=False)
        test_dataset = EEGMatrixDataset(x_eeg_path, labels_path, groups_path, metadata_path, split_indices["test"], is_train=False)
    else:
        transform = build_image_transform(model_config.get("image_size", 224))
        train_dataset = EEGImageDataset(x_images_path, labels_path, groups_path, metadata_path, split_indices["train"], transform=transform)
        val_dataset = EEGImageDataset(x_images_path, labels_path, groups_path, metadata_path, split_indices["val"], transform=transform)
        test_dataset = EEGImageDataset(x_images_path, labels_path, groups_path, metadata_path, split_indices["test"], transform=transform)

    # Weighted sampling reduces the chance that one class dominates mini-batches.
    train_labels = np.asarray(train_dataset.get_labels(), dtype=np.int64)
    class_counts = np.bincount(train_labels, minlength=2)
    sample_weights = 1.0 / np.maximum(class_counts[train_labels], 1)
    train_sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

    return train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader


In [45]:
# These are the CNN model definitions used in the notebook.
# Some are EEG specific models and some are image style CNN backbones.

class ConvBNAct(nn.Module):
    # This is a small reusable block made of convolution, batch norm, and activation.
    # Many CNNs in the notebook are built from this block.
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, act="relu"):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)

        if act == "relu":
            self.act = nn.ReLU(inplace=True)
        elif act == "leaky_relu":
            self.act = nn.LeakyReLU(0.1, inplace=True)
        elif act == "silu":
            self.act = nn.SiLU(inplace=True)
        else:
            raise ValueError(f"Unsupported activation: {act}")

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class SEBlock(nn.Module):
    # Squeeze excitation learns how important each feature channel is.
    # It lets the network emphasize useful channels and reduce weaker ones.
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        pooled = self.pool(x).view(b, c)
        scale = self.fc(pooled).view(b, c, 1, 1)
        return x * scale


class ResidualBlock(nn.Module):
    # Residual blocks help deeper networks train by adding a skip connection.
    # The block learns a correction to the input instead of learning everything from scratch.
    def __init__(self, channels, dropout=0.10):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNAct(channels, channels, kernel_size=3, padding=1, act="silu"),
            nn.Dropout2d(dropout),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(x + self.block(x))


def initialize_custom_model(module):
    # This initialization follows common deep learning practice.
    # Kaiming init works well with ReLU style activations.
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
            if getattr(m, "weight", None) is not None:
                nn.init.ones_(m.weight)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias)


class BaselineEEGCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # This is the simplest EEG model in the notebook.
        # It treats each epoch like a small 2D map where height is channels and width is time.
        # The goal is to give a clean baseline that is easy to compare against stronger models.
        self.features = nn.Sequential(
            ConvBNAct(1, 16, kernel_size=(5, 25), padding=(2, 12), act="relu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
            ConvBNAct(16, 32, kernel_size=(3, 15), padding=(1, 7), act="relu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
            ConvBNAct(32, 64, kernel_size=(3, 7), padding=(1, 3), act="relu"),
            nn.MaxPool2d(kernel_size=(2, 2)),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.55),
            nn.Linear(128, num_classes),
        )
        initialize_custom_model(self)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


class ResidualEEGCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # This is a deeper EEG specific CNN.
        # It adds residual connections, squeeze excitation, and a stronger classifier head.
        # It is meant to test whether a deeper network can learn better EEG features than the baseline model.
        self.stem = nn.Sequential(
            ConvBNAct(1, 32, kernel_size=(7, 31), padding=(3, 15), act="leaky_relu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
            ConvBNAct(32, 64, kernel_size=(5, 15), padding=(2, 7), act="silu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
        )
        self.res_stack = nn.Sequential(
            ResidualBlock(64, dropout=0.10),
            SEBlock(64),
            ConvBNAct(64, 128, kernel_size=3, padding=1, act="silu"),
            nn.MaxPool2d(kernel_size=(2, 2)),
            ResidualBlock(128, dropout=0.15),
            SEBlock(128),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.SiLU(inplace=True),
            nn.Dropout(0.55),
            nn.Linear(256, num_classes),
        )
        initialize_custom_model(self)

    def forward(self, x):
        x = self.stem(x)
        x = self.res_stack(x)
        return self.head(x)


class TwoBranchEEGCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # This model has two branches with different jobs.
        # One branch looks more at temporal patterns and the other looks more at channel relationships.
        # The two feature sets are fused before classification.
        # Branch 1 uses long kernels over time to capture temporal patterns.
        self.time_branch = nn.Sequential(
            ConvBNAct(1, 32, kernel_size=(3, 31), padding=(1, 15), act="relu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
            ConvBNAct(32, 64, kernel_size=(3, 15), padding=(1, 7), act="relu"),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # Branch 2 uses tall kernels over channels to capture spatial sensor relations.
        self.channel_branch = nn.Sequential(
            ConvBNAct(1, 32, kernel_size=(9, 7), padding=(4, 3), act="leaky_relu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
            ConvBNAct(32, 64, kernel_size=(7, 5), padding=(3, 2), act="leaky_relu"),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.SiLU(inplace=True),
            nn.Dropout(0.50),
            nn.Linear(128, num_classes),
        )
        initialize_custom_model(self)

    def forward(self, x):
        time_features = self.time_branch(x).flatten(1)
        channel_features = self.channel_branch(x).flatten(1)
        fused = torch.cat([time_features, channel_features], dim=1)
        return self.head(fused)


In [46]:
# These image backbone helpers try to use official torchvision weights first.
# If torchvision is not usable in the current environment, the code falls back to local CNNs with similar design ideas.

class BasicResBlock(nn.Module):
    # This is a small ResNet style block used by the fallback ResNet model.
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU(inplace=True)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.act(out + identity)
        return out


class FallbackResNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # This is a local ResNet like CNN used only when torchvision cannot load the official model.
        # It keeps the main ResNet idea of repeated residual blocks.
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        self.layers = nn.Sequential(
            BasicResBlock(32, 32, stride=1),
            BasicResBlock(32, 64, stride=2),
            BasicResBlock(64, 128, stride=2),
            BasicResBlock(128, 256, stride=2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)
        initialize_custom_model(self)

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


class DenseLayer(nn.Module):
    # DenseNet layers concatenate new features with old features.
    # This encourages strong feature reuse.
    def __init__(self, in_channels, growth_rate):
        super().__init__()
        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, growth_rate * 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(growth_rate * 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(growth_rate * 4, growth_rate, kernel_size=3, padding=1, bias=False),
        )

    def forward(self, x):
        new_features = self.layer(x)
        return torch.cat([x, new_features], dim=1)


class DenseBlock(nn.Module):
    # A dense block is a stack of dense layers.
    def __init__(self, in_channels, num_layers, growth_rate):
        super().__init__()
        layers = []
        channels = in_channels
        for _ in range(num_layers):
            layers.append(DenseLayer(channels, growth_rate))
            channels += growth_rate
        self.block = nn.Sequential(*layers)
        self.out_channels = channels

    def forward(self, x):
        return self.block(x)


class TransitionLayer(nn.Module):
    # Transition layers reduce the feature size between dense blocks.
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )

    def forward(self, x):
        return self.layer(x)


class MBConvBlock(nn.Module):
    # This is an EfficientNet style mobile inverted bottleneck block.
    def __init__(self, in_channels, out_channels, expand_ratio=4, stride=1):
        super().__init__()
        hidden = in_channels * expand_ratio
        self.use_residual = stride == 1 and in_channels == out_channels
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, hidden, kernel_size=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, hidden, kernel_size=3, stride=stride, padding=1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.se = SEBlock(out_channels)

    def forward(self, x):
        out = self.se(self.block(x))
        if self.use_residual:
            out = out + x
        return out


class FallbackEfficientNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # This is a local EfficientNet like CNN used if the official torchvision model is unavailable.
        # It uses MBConv style blocks for a better accuracy and efficiency tradeoff.
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(inplace=True),
        )
        self.blocks = nn.Sequential(
            MBConvBlock(32, 32, expand_ratio=1, stride=1),
            MBConvBlock(32, 48, expand_ratio=4, stride=2),
            MBConvBlock(48, 64, expand_ratio=4, stride=2),
            MBConvBlock(64, 96, expand_ratio=4, stride=2),
            MBConvBlock(96, 128, expand_ratio=4, stride=1),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)
        initialize_custom_model(self)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


class LayerNorm2d(nn.Module):
    # ConvNeXt style models often use layer normalization rather than batch normalization.
    def __init__(self, num_channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=1, keepdim=True)
        var = (x - mean).pow(2).mean(dim=1, keepdim=True)
        x = (x - mean) / torch.sqrt(var + self.eps)
        return self.weight[:, None, None] * x + self.bias[:, None, None]


class ConvNeXtBlock(nn.Module):
    # This is a ConvNeXt style block with depthwise convolution and pointwise projections.
    def __init__(self, channels, drop_prob=0.0):
        super().__init__()
        self.dwconv = nn.Conv2d(channels, channels, kernel_size=7, padding=3, groups=channels)
        self.norm = LayerNorm2d(channels)
        self.pw1 = nn.Conv2d(channels, 4 * channels, kernel_size=1)
        self.act = nn.GELU()
        self.pw2 = nn.Conv2d(4 * channels, channels, kernel_size=1)
        self.drop_prob = drop_prob

    def forward(self, x):
        out = self.dwconv(x)
        out = self.norm(out)
        out = self.pw1(out)
        out = self.act(out)
        out = self.pw2(out)
        if self.training and self.drop_prob > 0.0:
            keep_prob = 1.0 - self.drop_prob
            mask = torch.rand((out.shape[0], 1, 1, 1), device=out.device) < keep_prob
            out = out * mask / keep_prob
        return x + out


class FallbackConvNeXt(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # This is a local ConvNeXt like CNN used if the official ConvNeXt cannot be loaded.
        # It keeps the main idea of modern pure CNN design.
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=4),
            LayerNorm2d(64),
        )
        self.stage1 = nn.Sequential(ConvNeXtBlock(64), ConvNeXtBlock(64))
        self.down1 = nn.Sequential(LayerNorm2d(64), nn.Conv2d(64, 128, kernel_size=2, stride=2))
        self.stage2 = nn.Sequential(ConvNeXtBlock(128), ConvNeXtBlock(128))
        self.down2 = nn.Sequential(LayerNorm2d(128), nn.Conv2d(128, 256, kernel_size=2, stride=2))
        self.stage3 = nn.Sequential(ConvNeXtBlock(256), ConvNeXtBlock(256))
        self.norm = LayerNorm2d(256)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)
        initialize_custom_model(self)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.down1(x)
        x = self.stage2(x)
        x = self.down2(x)
        x = self.stage3(x)
        x = self.norm(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


class RegNetBlock(nn.Module):
    # This is a RegNet style block with grouped convolution and squeeze excitation.
    def __init__(self, in_channels, out_channels, stride=1, groups=8):
        super().__init__()
        inner = out_channels
        self.proj = None
        if stride != 1 or in_channels != out_channels:
            self.proj = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, inner, kernel_size=1, bias=False),
            nn.BatchNorm2d(inner),
            nn.ReLU(inplace=True),
            nn.Conv2d(inner, inner, kernel_size=3, stride=stride, padding=1, groups=min(groups, inner), bias=False),
            nn.BatchNorm2d(inner),
            nn.ReLU(inplace=True),
            nn.Conv2d(inner, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.se = SEBlock(out_channels)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x if self.proj is None else self.proj(x)
        out = self.se(self.block(x))
        return self.act(out + identity)


class FallbackRegNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # This is a local RegNet like CNN used if the official RegNet cannot be loaded.
        # It gives another image backbone with a different design bias from ResNet and EfficientNet.
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.layers = nn.Sequential(
            RegNetBlock(32, 48, stride=2, groups=8),
            RegNetBlock(48, 96, stride=2, groups=8),
            RegNetBlock(96, 160, stride=2, groups=16),
            RegNetBlock(160, 224, stride=1, groups=16),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(224, num_classes)
        initialize_custom_model(self)

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


def get_resnet18_model(num_classes=2, pretrained=True):
    # ResNet-18 is a standard transfer learning baseline.
    # It uses residual skip connections and is usually stable to fine tune.
    try:
        from torchvision.models import ResNet18_Weights, resnet18

        weights = ResNet18_Weights.DEFAULT if pretrained else None
        model = resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model
    except Exception as exc:
        print(f"Falling back to local ResNet-like model because torchvision is unavailable: {exc}")
        return FallbackResNet(num_classes=num_classes)


def get_densenet121_model(num_classes=2, pretrained=True):
    # DenseNet-121 reuses earlier features through concatenation.
    # This can help when the signal is subtle and low level details matter.
    try:
        from torchvision.models import DenseNet121_Weights, densenet121

        weights = DenseNet121_Weights.DEFAULT if pretrained else None
        model = densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
        return model
    except Exception as exc:
        print(f"Falling back to local DenseNet-like model because torchvision is unavailable: {exc}")
        growth_rate = 16
        stem_channels = 32

        block1 = DenseBlock(stem_channels, num_layers=4, growth_rate=growth_rate)
        trans1_out = 64
        block2 = DenseBlock(trans1_out, num_layers=4, growth_rate=growth_rate)
        trans2_out = 96
        block3 = DenseBlock(trans2_out, num_layers=4, growth_rate=growth_rate)
        final_channels = block3.out_channels

        class FallbackDenseNet(nn.Module):
            def __init__(self):
                super().__init__()
                self.stem = nn.Sequential(
                    nn.Conv2d(3, stem_channels, kernel_size=7, stride=2, padding=3, bias=False),
                    nn.BatchNorm2d(stem_channels),
                    nn.ReLU(inplace=True),
                    nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
                )
                self.block1 = block1
                self.trans1 = TransitionLayer(block1.out_channels, trans1_out)
                self.block2 = block2
                self.trans2 = TransitionLayer(block2.out_channels, trans2_out)
                self.block3 = block3
                self.norm = nn.BatchNorm2d(final_channels)
                self.pool = nn.AdaptiveAvgPool2d((1, 1))
                self.fc = nn.Linear(final_channels, num_classes)
                initialize_custom_model(self)

            def forward(self, x):
                x = self.stem(x)
                x = self.block1(x)
                x = self.trans1(x)
                x = self.block2(x)
                x = self.trans2(x)
                x = self.block3(x)
                x = self.norm(x)
                x = torch.relu(x)
                x = self.pool(x).flatten(1)
                return self.fc(x)

        return FallbackDenseNet()


def get_efficientnet_b0_model(num_classes=2, pretrained=True):
    # EfficientNet-B0 is a compact CNN with a strong accuracy and efficiency tradeoff.
    try:
        from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = efficientnet_b0(weights=weights)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
        return model
    except Exception as exc:
        print(f"Falling back to local EfficientNet-like model because torchvision is unavailable: {exc}")
        return FallbackEfficientNet(num_classes=num_classes)


def get_convnext_tiny_model(num_classes=2, pretrained=True):
    # ConvNeXt-Tiny is a modern pure CNN backbone.
    # It is useful to test whether a newer CNN family transfers better to the EEG image view.
    try:
        from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny

        weights = ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        model = convnext_tiny(weights=weights)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
        return model
    except Exception as exc:
        print(f"Falling back to local ConvNeXt-like model because torchvision is unavailable: {exc}")
        return FallbackConvNeXt(num_classes=num_classes)


def get_regnet_y_400mf_model(num_classes=2, pretrained=True):
    # RegNetY-400MF is another efficient pretrained CNN family.
    # It gives a different scaling and design choice from ResNet and EfficientNet.
    try:
        from torchvision.models import RegNet_Y_400MF_Weights, regnet_y_400mf

        weights = RegNet_Y_400MF_Weights.DEFAULT if pretrained else None
        model = regnet_y_400mf(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model
    except Exception as exc:
        print(f"Falling back to local RegNet-like model because torchvision is unavailable: {exc}")
        return FallbackRegNet(num_classes=num_classes)


def build_model(model_config):
    # This factory chooses the right model based on the config entry.
    # The comments below summarize the role of each model.
    model_name = model_config["name"]
    if model_name == "baseline_eeg_cnn":
        # Simple direct EEG baseline.
        return BaselineEEGCNN(num_classes=2)
    if model_name == "residual_eeg_cnn":
        # Deeper EEG model with residual and channel attention ideas.
        return ResidualEEGCNN(num_classes=2)
    if model_name == "two_branch_eeg_cnn":
        # EEG model that separates temporal and channel processing before fusion.
        return TwoBranchEEGCNN(num_classes=2)
    if model_name == "resnet18_finetune":
        # Standard pretrained image CNN baseline.
        return get_resnet18_model(num_classes=2, pretrained=model_config.get("pretrained", True))
    if model_name == "densenet121_finetune":
        # Image CNN with dense feature reuse.
        return get_densenet121_model(num_classes=2, pretrained=model_config.get("pretrained", True))
    if model_name == "efficientnet_b0_finetune":
        # Compact and efficient pretrained style image CNN.
        return get_efficientnet_b0_model(num_classes=2, pretrained=model_config.get("pretrained", True))
    if model_name == "convnext_tiny_finetune":
        # Modern pure CNN image backbone.
        return get_convnext_tiny_model(num_classes=2, pretrained=model_config.get("pretrained", True))
    if model_name == "regnet_y_400mf_finetune":
        # Efficient image backbone from the RegNet family.
        return get_regnet_y_400mf_model(num_classes=2, pretrained=model_config.get("pretrained", True))
    raise ValueError(f"Unknown model name: {model_name}")


In [47]:
# This table controls the full experiment list.
# Each row is one model run.
# The list is intentionally small now so the notebook matches the strongest results we have seen so far.
# This keeps the comparison focused on the best EEG models for task 1.

MODEL_CONFIGS = [
    {
        # Stronger EEG model with residual blocks and light temporal augmentation.
        "name": "residual_eeg_cnn",
        "input_type": "eeg",
        "batch_size": 24,
        "num_workers": 0,
        "lr": 2e-4,
        "weight_decay": 2e-3,
        "label_smoothing": 0.10,
        "epochs": 35,
        "patience": 8,
        "monitor_metric": "balanced_accuracy",
        "loss_name": "cross_entropy",
        "time_jitter_max": 10,
        "time_crop_fraction": 0.94,
        "same_class_mixup_alpha": 0.0,
        "use_subject_cv": False,
    },
    {
        # EEG model with separate temporal and channel branches.
        "name": "two_branch_eeg_cnn",
        "input_type": "eeg",
        "batch_size": 24,
        "num_workers": 0,
        "lr": 2e-4,
        "weight_decay": 2e-3,
        "label_smoothing": 0.12,
        "epochs": 40,
        "patience": 10,
        "monitor_metric": "balanced_accuracy",
        "loss_name": "cross_entropy",
        "time_jitter_max": 10,
        "time_crop_fraction": 0.94,
        "same_class_mixup_alpha": 0.0,
        "use_subject_cv": False,
    },
    {
        # Subject-wise 5-fold cross-validation run for the best current model.
        "name": "residual_eeg_cnn",
        "run_name": "residual_eeg_cnn_cv5",
        "input_type": "eeg",
        "batch_size": 24,
        "num_workers": 0,
        "lr": 2e-4,
        "weight_decay": 2e-3,
        "label_smoothing": 0.10,
        "epochs": 35,
        "patience": 8,
        "monitor_metric": "balanced_accuracy",
        "loss_name": "cross_entropy",
        "time_jitter_max": 10,
        "time_crop_fraction": 0.94,
        "same_class_mixup_alpha": 0.0,
        "use_subject_cv": True,
        "num_cv_folds": 5,
    },
]

pd.DataFrame(MODEL_CONFIGS)


,name,input_type,batch_size,num_workers,lr,weight_decay,label_smoothing,epochs,patience,monitor_metric,loss_name,time_jitter_max,time_crop_fraction,same_class_mixup_alpha,use_subject_cv,run_name,num_cv_folds
0,residual_eeg_cnn,eeg,24,0,0.0002,0.002,0.10,35,8,balanced_accuracy,cross_entropy,10,0.94,0.0,False,NaN,NaN
1,two_branch_eeg_cnn,eeg,24,0,0.0002,0.002,0.12,40,10,balanced_accuracy,cross_entropy,10,0.94,0.0,False,NaN,NaN
2,residual_eeg_cnn,eeg,24,0,0.0002,0.002,0.10,35,8,balanced_accuracy,cross_entropy,10,0.94,0.0,True,residual_eeg_cnn_cv5,5.0


In [48]:
# This cell does a quick smoke test.
# It builds each configured model once and checks that a forward pass works.
# This is useful before starting long training runs.

SMOKE_TEST_MODELS = [cfg["name"] for cfg in MODEL_CONFIGS]

for cfg in MODEL_CONFIGS:
    if cfg["name"] not in SMOKE_TEST_MODELS:
        continue
    train_dataset, _, _, train_loader, _, _ = build_dataloaders(cfg)
    model = build_model(cfg).to(DEVICE)
    batch = next(iter(train_loader))
    images, labels, metas = batch
    with torch.no_grad():
        logits = model(images.to(DEVICE))
    print(f"{cfg['name']}: input={tuple(images.shape)}, logits={tuple(logits.shape)}")


residual_eeg_cnn: input=(24, 1, 61, 1250), logits=(24, 2)
two_branch_eeg_cnn: input=(24, 1, 61, 1250), logits=(24, 2)
residual_eeg_cnn: input=(24, 1, 61, 1250), logits=(24, 2)


In [49]:
# This cell contains shared training and evaluation helpers.
# It includes focal loss, same-class mixup, and session-level aggregation.

def compute_class_weights(labels, num_classes=2):
    labels = np.asarray(labels, dtype=np.int64)
    class_counts = np.bincount(labels, minlength=num_classes)
    class_weights = len(labels) / (num_classes * np.maximum(class_counts, 1))
    return torch.tensor(class_weights, dtype=torch.float32, device=DEVICE), class_counts


def compute_metrics(y_true, y_pred):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="binary"),
    }
    return metrics


def compute_metrics_with_probabilities(y_true, y_pred, y_prob):
    # This version also adds AUROC so you can compare runs with the first notebook more directly.
    metrics = compute_metrics(y_true, y_pred)
    try:
        metrics["auroc"] = roc_auc_score(y_true, y_prob)
    except ValueError:
        metrics["auroc"] = float("nan")
    return metrics


class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        loss = ((1.0 - pt) ** self.gamma) * ce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


def build_loss_function(model_config, class_weights):
    loss_name = model_config.get("loss_name", "cross_entropy")
    if loss_name == "focal":
        return FocalLoss(weight=class_weights, gamma=model_config.get("focal_gamma", 2.0))
    return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=model_config.get("label_smoothing", 0.0))


def apply_same_class_mixup(inputs, labels, alpha):
    if alpha is None or alpha <= 0.0:
        return inputs

    mixed = inputs.clone()
    unique_labels = labels.unique()

    for class_id in unique_labels.tolist():
        class_positions = torch.where(labels == class_id)[0]
        if len(class_positions) < 2:
            continue
        shuffled = class_positions[torch.randperm(len(class_positions), device=labels.device)]
        lam = float(np.random.beta(alpha, alpha))
        mixed[class_positions] = lam * inputs[class_positions] + (1.0 - lam) * inputs[shuffled]

    return mixed


def make_session_level_outputs(epoch_outputs):
    # Session-level aggregation is usually a better match for the real task than single-epoch decisions.
    # We average the sleep-deprivation probability over all epochs from the same recording session.
    epoch_df = pd.DataFrame({
        "global_index": epoch_outputs["global_index"],
        "y_true": epoch_outputs["y_true"],
        "y_pred": epoch_outputs["y_pred"],
        "y_prob_sd": epoch_outputs["y_prob"],
    }).merge(
        metadata.assign(global_index=np.arange(len(metadata))),
        on="global_index",
        how="left",
    )

    session_df = (
        epoch_df
        .groupby(["subject_id", "session", "file_path"], as_index=False)
        .agg(
            y_true=("y_true", "first"),
            y_prob_sd=("y_prob_sd", "mean"),
            num_epochs=("global_index", "count"),
        )
    )
    session_df["y_pred"] = (session_df["y_prob_sd"] >= 0.5).astype(int)

    session_metrics = compute_metrics(session_df["y_true"].to_numpy(), session_df["y_pred"].to_numpy())
    session_report = classification_report(
        session_df["y_true"].to_numpy(),
        session_df["y_pred"].to_numpy(),
        target_names=["NS", "SD"],
        output_dict=True,
        zero_division=0,
    )
    session_cm = confusion_matrix(session_df["y_true"].to_numpy(), session_df["y_pred"].to_numpy())

    return {
        "session_df": session_df,
        "metrics": session_metrics,
        "report": session_report,
        "confusion_matrix": session_cm,
    }


def choose_monitor_metrics(epoch_outputs):
    # This helper returns both epoch-level and session-level metrics from the validation split.
    # We select checkpoints using the session-level metric to reduce overfitting to noisy individual epochs.
    session_outputs = make_session_level_outputs(epoch_outputs)
    return epoch_outputs["metrics"], session_outputs["metrics"], session_outputs


def evaluate_model(model, loader, criterion, device, split_name="val"):
    # This runs evaluation without gradient updates.
    # It returns loss, predictions, probabilities, and classification metrics.
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []
    all_indices = []

    with torch.no_grad():
        for images, labels, metas in tqdm(loader, desc=f"{split_name}", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)

            running_loss += loss.item() * images.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_indices.extend(metas["global_index"])

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_preds)
    y_prob = np.asarray(all_probs)

    return {
        "loss": running_loss / len(loader.dataset),
        "metrics": compute_metrics_with_probabilities(y_true, y_pred, y_prob),
        "report": classification_report(y_true, y_pred, target_names=["NS", "SD"], output_dict=True, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "global_index": np.asarray(all_indices),
    }


def train_one_epoch(model, loader, criterion, optimizer, device, epoch, num_epochs, same_class_mixup_alpha=0.0):
    # This runs one training epoch.
    # It can apply same-class mixup before the model sees the batch.
    model.train()
    running_loss = 0.0
    all_labels = []
    all_preds = []

    progress = tqdm(loader, desc=f"train {epoch}/{num_epochs}", leave=False)
    for images, labels, _ in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        images = apply_same_class_mixup(images, labels, alpha=same_class_mixup_alpha)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        preds = torch.argmax(logits, dim=1)
        running_loss += loss.item() * images.size(0)
        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())

        progress.set_postfix(loss=f"{running_loss / max(len(all_labels), 1):.4f}")

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_preds)
    return running_loss / len(loader.dataset), compute_metrics(y_true, y_pred)


In [50]:
# This cell runs a full train, validation, and test cycle for one config.
# It also saves epoch-level and session-level outputs to disk.

def run_experiment(model_config):
    run_name = model_config.get("run_name", model_config["name"])
    # Each experiment gets its own folder so results do not overwrite each other.
    experiment_dir = EXPERIMENT_ROOT / run_name
    checkpoint_path = experiment_dir / "best_model.pt"
    experiment_dir.mkdir(parents=True, exist_ok=True)

    # Build the datasets and loaders for this model.
    train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader = build_dataloaders(model_config)
    class_weights, class_counts = compute_class_weights(train_dataset.get_labels())

    # Build the model and training pieces from the config.
    model = build_model(model_config).to(DEVICE)
    criterion = build_loss_function(model_config, class_weights)
    optimizer = AdamW(model.parameters(), lr=model_config["lr"], weight_decay=model_config["weight_decay"])
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    history = []
    best_metric = -np.inf
    best_state = None
    best_epoch = -1
    stale_epochs = 0

    config_to_save = dict(model_config)
    config_to_save["train_class_counts"] = class_counts.tolist()
    with open(experiment_dir / "run_config.json", "w", encoding="utf-8") as f:
        json.dump(config_to_save, f, indent=2)

    start_time = time.time()

    # Main training loop with validation after each epoch.
    # The checkpoint is selected from validation performance.
    # This matches the workflow in the first notebook more closely.
    for epoch in range(1, model_config["epochs"] + 1):
        train_loss, train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE, epoch, model_config["epochs"],
            same_class_mixup_alpha=model_config.get("same_class_mixup_alpha", 0.0),
        )
        val_outputs = evaluate_model(model, val_loader, criterion, DEVICE, split_name="val")
        val_loss = val_outputs["loss"]
        val_metrics, val_session_metrics, val_session_outputs = choose_monitor_metrics(val_outputs)
        current_metric = val_session_metrics[model_config["monitor_metric"]]
        scheduler.step(current_metric)

        history_row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": train_loss,
            "val_loss": val_loss,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
            **{f"val_session_{k}": v for k, v in val_session_metrics.items()},
        }
        history.append(history_row)
        pd.DataFrame(history).to_csv(experiment_dir / "training_history.csv", index=False)

        improved = current_metric > best_metric
        if improved:
            best_metric = current_metric
            best_epoch = epoch
            stale_epochs = 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save({"model_state_dict": best_state, "epoch": epoch, "best_metric": best_metric}, checkpoint_path)
        else:
            stale_epochs += 1

        print(
            f"[{run_name}] epoch {epoch:02d}/{model_config['epochs']} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | val_f1={val_metrics['f1']:.4f} | "
            f"val_bal_acc={val_metrics['balanced_accuracy']:.4f} | "
            f"val_auroc={val_metrics['auroc']:.4f} | "
            f"val_session_f1={val_session_metrics['f1']:.4f} | "
            f"val_session_bal_acc={val_session_metrics['balanced_accuracy']:.4f} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

        if stale_epochs >= model_config["patience"]:
            print(f"Early stopping for {run_name} at epoch {epoch}.")
            break

    if best_state is None:
        raise RuntimeError(f"No checkpoint was saved for {run_name}.")

    # Load the best validation checkpoint before saving the comparison summary.
    model.load_state_dict(best_state)
    val_outputs = evaluate_model(model, val_loader, criterion, DEVICE, split_name="val_best")
    val_session_outputs = make_session_level_outputs(val_outputs)

    elapsed_minutes = (time.time() - start_time) / 60.0

    val_predictions = pd.DataFrame({
        "global_index": val_outputs["global_index"],
        "y_true": val_outputs["y_true"],
        "y_pred": val_outputs["y_pred"],
        "y_prob_sd": val_outputs["y_prob"],
    }).merge(
        metadata.assign(global_index=np.arange(len(metadata))),
        on="global_index",
        how="left",
    )
    val_predictions.to_csv(experiment_dir / "best_val_predictions.csv", index=False)
    val_session_outputs["session_df"].to_csv(experiment_dir / "val_session_predictions.csv", index=False)

    # Save a compact summary for model comparison.
    # Test is intentionally left out here so the held out test set stays untouched during tuning.
    summary = {
        "model_name": model_config["name"],
        "run_name": run_name,
        "best_epoch": best_epoch,
        "best_val_metric": float(best_metric),
        "monitor_metric": model_config["monitor_metric"],
        "elapsed_minutes": elapsed_minutes,
        "test_evaluated": False,
        "val_metrics": {k: float(v) for k, v in val_outputs["metrics"].items()},
        "val_session_metrics": {k: float(v) for k, v in val_session_outputs["metrics"].items()},
        "val_confusion_matrix": val_outputs["confusion_matrix"].tolist(),
        "val_session_confusion_matrix": val_session_outputs["confusion_matrix"].tolist(),
        "val_classification_report": val_outputs["report"],
        "val_session_classification_report": val_session_outputs["report"],
        "test_metrics": None,
        "test_session_metrics": None,
        "test_confusion_matrix": None,
        "test_session_confusion_matrix": None,
        "test_classification_report": None,
        "test_session_classification_report": None,
    }

    with open(experiment_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    return summary


def run_final_test_evaluation(selected_run_name):
    # This runs the held out test only after you choose the final model from validation.
    # It keeps the project proposal workflow clean.
    matching_configs = [
        cfg for cfg in MODEL_CONFIGS
        if not cfg.get("use_subject_cv", False) and cfg.get("run_name", cfg["name"]) == selected_run_name
    ]
    if len(matching_configs) != 1:
        raise ValueError(f"Could not find one standard config for run name: {selected_run_name}")

    model_config = matching_configs[0]
    experiment_dir = EXPERIMENT_ROOT / selected_run_name
    checkpoint_path = experiment_dir / "best_model.pt"
    summary_path = experiment_dir / "summary.json"

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Best checkpoint is missing for {selected_run_name}: {checkpoint_path}")
    if not summary_path.exists():
        raise FileNotFoundError(f"Summary file is missing for {selected_run_name}: {summary_path}")

    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    _, _, test_dataset, _, _, test_loader = build_dataloaders(model_config)
    train_dataset, _, _, _, _, _ = build_dataloaders(model_config)
    class_weights, _ = compute_class_weights(train_dataset.get_labels())
    criterion = build_loss_function(model_config, class_weights)
    model = build_model(model_config).to(DEVICE)

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    test_outputs = evaluate_model(model, test_loader, criterion, DEVICE, split_name="test_final")
    test_session_outputs = make_session_level_outputs(test_outputs)

    test_predictions = pd.DataFrame({
        "global_index": test_outputs["global_index"],
        "y_true": test_outputs["y_true"],
        "y_pred": test_outputs["y_pred"],
        "y_prob_sd": test_outputs["y_prob"],
    }).merge(
        metadata.assign(global_index=np.arange(len(metadata))),
        on="global_index",
        how="left",
    )
    test_predictions.to_csv(experiment_dir / "final_test_predictions.csv", index=False)
    test_session_outputs["session_df"].to_csv(experiment_dir / "final_test_session_predictions.csv", index=False)

    summary["test_evaluated"] = True
    summary["test_metrics"] = {k: float(v) for k, v in test_outputs["metrics"].items()}
    summary["test_session_metrics"] = {k: float(v) for k, v in test_session_outputs["metrics"].items()}
    summary["test_confusion_matrix"] = test_outputs["confusion_matrix"].tolist()
    summary["test_session_confusion_matrix"] = test_session_outputs["confusion_matrix"].tolist()
    summary["test_classification_report"] = test_outputs["report"]
    summary["test_session_classification_report"] = test_session_outputs["report"]

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    return summary


def run_subject_wise_cross_validation(model_config):
    run_name = model_config.get("run_name", model_config["name"])
    # This runs subject-wise cross-validation.
    # Each fold keeps subjects separated between train and validation.
    n_splits = int(model_config.get("num_cv_folds", NUM_CV_FOLDS))
    fold_results = []
    cv_dir = EXPERIMENT_ROOT / run_name
    cv_dir.mkdir(parents=True, exist_ok=True)

    for fold in cv_splits[:n_splits]:
        fold_config = dict(model_config)
        fold_config["run_name"] = f"{run_name}_fold_{fold['fold_id']}"
        fold_split_indices = {
            "train": fold["train"],
            "val": fold["val"],
            "test": fold["test"],
        }

        experiment_dir = EXPERIMENT_ROOT / fold_config["run_name"]
        checkpoint_path = experiment_dir / "best_model.pt"
        experiment_dir.mkdir(parents=True, exist_ok=True)

        train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader = build_dataloaders(fold_config, split_indices=fold_split_indices)
        class_weights, class_counts = compute_class_weights(train_dataset.get_labels())
        model = build_model(fold_config).to(DEVICE)
        criterion = build_loss_function(fold_config, class_weights)
        optimizer = AdamW(model.parameters(), lr=fold_config["lr"], weight_decay=fold_config["weight_decay"])
        scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

        best_metric = -np.inf
        best_state = None
        stale_epochs = 0

        for epoch in range(1, fold_config["epochs"] + 1):
            train_one_epoch(
                model, train_loader, criterion, optimizer, DEVICE, epoch, fold_config["epochs"],
                same_class_mixup_alpha=fold_config.get("same_class_mixup_alpha", 0.0),
            )
            val_outputs = evaluate_model(model, val_loader, criterion, DEVICE, split_name=f"cv_fold_{fold['fold_id']}_val")
            _, val_session_metrics, _ = choose_monitor_metrics(val_outputs)
            current_metric = val_session_metrics[fold_config["monitor_metric"]]
            scheduler.step(current_metric)

            if current_metric > best_metric:
                best_metric = current_metric
                best_state = copy.deepcopy(model.state_dict())
                stale_epochs = 0
                torch.save({"model_state_dict": best_state, "best_metric": best_metric}, checkpoint_path)
            else:
                stale_epochs += 1

            if stale_epochs >= fold_config["patience"]:
                break

        if best_state is None:
            raise RuntimeError(f"No checkpoint was saved for {fold_config['run_name']}.")

        model.load_state_dict(best_state)
        fold_val_outputs = evaluate_model(model, val_loader, criterion, DEVICE, split_name=f"cv_fold_{fold['fold_id']}_best")
        fold_session_outputs = make_session_level_outputs(fold_val_outputs)
        fold_results.append({
            "fold_id": fold["fold_id"],
            "epoch_metrics": fold_val_outputs["metrics"],
            "session_metrics": fold_session_outputs["metrics"],
        })

    cv_summary = {
        "model_name": model_config["name"],
        "run_name": run_name,
        "num_cv_folds": n_splits,
        "fold_results": fold_results,
        "mean_epoch_f1": float(np.mean([x["epoch_metrics"]["f1"] for x in fold_results])),
        "mean_session_f1": float(np.mean([x["session_metrics"]["f1"] for x in fold_results])),
    }

    with open(cv_dir / "cv_summary.json", "w", encoding="utf-8") as f:
        json.dump(cv_summary, f, indent=2)

    return cv_summary


In [51]:
# This is the main run cell.
# Standard experiments compare the strongest current EEG models on train and validation only.
# This follows the task 1 goal from the project proposal and the workflow in file 1.
# Cross-validation for the best model is available as a separate final check.

MODELS_TO_RUN = [
    "residual_eeg_cnn",
    "two_branch_eeg_cnn",
]
CV_MODELS_TO_RUN = [cfg.get("run_name", cfg["name"]) for cfg in MODEL_CONFIGS if cfg.get("use_subject_cv", False)]
RUN_STANDARD_EXPERIMENTS = True
RUN_CV_EXPERIMENTS = False

all_results = []
if RUN_STANDARD_EXPERIMENTS:
    for cfg in MODEL_CONFIGS:
        run_name = cfg.get("run_name", cfg["name"])
        if cfg.get("use_subject_cv", False):
            continue
        if run_name not in MODELS_TO_RUN:
            continue
        result = run_experiment(cfg)
        all_results.append(result)

    results_df = pd.DataFrame([
        {
            "run_name": item.get("run_name", item["model_name"]),
            "model_name": item["model_name"],
            "best_epoch": item["best_epoch"],
            "val_accuracy": item["val_metrics"]["accuracy"],
            "val_f1": item["val_metrics"]["f1"],
            "val_bal_acc": item["val_metrics"]["balanced_accuracy"],
            "val_auroc": item["val_metrics"]["auroc"],
            "val_session_f1": item["val_session_metrics"]["f1"],
            "val_session_bal_acc": item["val_session_metrics"]["balanced_accuracy"],
            "elapsed_minutes": item["elapsed_minutes"],
        }
        for item in all_results
    ]).sort_values(by="val_session_f1", ascending=False)
    display(results_df)
else:
    print("Standard experiment training is skipped.")

cv_results = []
if RUN_CV_EXPERIMENTS:
    for cfg in MODEL_CONFIGS:
        run_name = cfg.get("run_name", cfg["name"])
        if not cfg.get("use_subject_cv", False):
            continue
        if run_name not in CV_MODELS_TO_RUN:
            continue
        result = run_subject_wise_cross_validation(cfg)
        cv_results.append(result)

    cv_results_df = pd.DataFrame([
        {
            "run_name": item["run_name"],
            "model_name": item["model_name"],
            "num_cv_folds": item["num_cv_folds"],
            "mean_epoch_f1": item["mean_epoch_f1"],
            "mean_session_f1": item["mean_session_f1"],
        }
        for item in cv_results
    ]).sort_values(by="mean_epoch_f1", ascending=False)
    display(cv_results_df)
else:
    print("Subject-wise cross-validation is skipped.")


train 1/35:   0%|          | 0/240 [00:00<?, ?it/s]

val:   0%|          | 0/52 [00:00<?, ?it/s]

[residual_eeg_cnn] epoch 01/35 | train_loss=0.6820 | val_loss=0.6625 | val_acc=0.6172 | val_f1=0.6125 | val_bal_acc=0.6177 | val_auroc=0.7082 | val_session_f1=0.6000 | val_session_bal_acc=0.6182 | lr=2.00e-04


train 2/35:   0%|          | 0/240 [00:00<?, ?it/s]

val:   0%|          | 0/52 [00:00<?, ?it/s]

...

val:   0%|          | 0/52 [00:00<?, ?it/s]

[two_branch_eeg_cnn] epoch 21/40 | train_loss=0.4936 | val_loss=0.6283 | val_acc=0.7632 | val_f1=0.7657 | val_bal_acc=0.7650 | val_auroc=0.8179 | val_session_f1=0.8571 | val_session_bal_acc=0.8591 | lr=6.25e-06
Early stopping for two_branch_eeg_cnn at epoch 21.


val_best:   0%|          | 0/52 [00:00<?, ?it/s]

,run_name,model_name,best_epoch,val_accuracy,val_f1,val_bal_acc,val_auroc,val_session_f1,val_session_bal_acc,elapsed_minutes
1,two_branch_eeg_cnn,two_branch_eeg_cnn,11,0.770465,0.753873,0.768991,0.812953,0.900000,0.904545,2.042535
0,residual_eeg_cnn,residual_eeg_cnn,2,0.711075,0.717425,0.713256,0.747541,0.761905,0.763636,0.903677


Subject-wise cross-validation is skipped.


In [52]:
# This cell runs the held out test set for both final CNN candidates.

FINAL_MODEL_RUN_NAMES = [
    "residual_eeg_cnn",
    "two_branch_eeg_cnn",
]
FINAL_SELECTED_MODEL = "residual_eeg_cnn"
RUN_FINAL_TEST_EVALUATION = True

if RUN_FINAL_TEST_EVALUATION:
    final_test_rows = []
    final_test_summaries = {}

    for run_name in FINAL_MODEL_RUN_NAMES:
        summary = run_final_test_evaluation(run_name)
        final_test_summaries[run_name] = summary
        final_test_rows.append({
            "run_name": summary["run_name"],
            "model_name": summary["model_name"],
            "test_accuracy": summary["test_metrics"]["accuracy"],
            "test_f1": summary["test_metrics"]["f1"],
            "test_bal_acc": summary["test_metrics"]["balanced_accuracy"],
            "test_auroc": summary["test_metrics"]["auroc"],
            "test_session_f1": summary["test_session_metrics"]["f1"],
            "test_session_bal_acc": summary["test_session_metrics"]["balanced_accuracy"],
        })

    final_test_df = pd.DataFrame(final_test_rows).sort_values(by="test_f1", ascending=False)
    display(final_test_df)

    print("Final selected model for Task 1:", FINAL_SELECTED_MODEL)
else:
    print("Final test evaluation is skipped.")


test_final:   0%|          | 0/55 [00:00<?, ?it/s]

test_final:   0%|          | 0/55 [00:00<?, ?it/s]

,run_name,model_name,test_accuracy,test_f1,test_bal_acc,test_auroc,test_session_f1,test_session_bal_acc
0,residual_eeg_cnn,residual_eeg_cnn,0.648176,0.616404,0.647931,0.715865,0.700000,0.727273
1,two_branch_eeg_cnn,two_branch_eeg_cnn,0.607143,0.526990,0.606633,0.668960,0.631579,0.681818


Final selected model for Task 1: residual_eeg_cnn


In [53]:
# This cell saves the final selected PyTorch model file.
# The final selected model is fixed as residual_eeg_cnn.

import json
from pathlib import Path
import torch

FINAL_SELECTED_MODEL = "residual_eeg_cnn"

save_dir = EXPERIMENT_ROOT / "final_selected_model"
save_dir.mkdir(parents=True, exist_ok=True)

summary_path = EXPERIMENT_ROOT / FINAL_SELECTED_MODEL / "summary.json"
checkpoint_path = EXPERIMENT_ROOT / FINAL_SELECTED_MODEL / "best_model.pt"

if not summary_path.exists():
    raise FileNotFoundError(f"Missing summary file: {summary_path}")

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Missing checkpoint file: {checkpoint_path}")

with open(summary_path, "r", encoding="utf-8") as f:
    best_summary = json.load(f)

best_config = None
for cfg in MODEL_CONFIGS:
    run_name = cfg.get("run_name", cfg["name"])
    if run_name == FINAL_SELECTED_MODEL:
        best_config = cfg
        break

if best_config is None:
    raise ValueError(f"Could not find config for run: {FINAL_SELECTED_MODEL}")

model = build_model(best_config).to(DEVICE)

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

state_dict_path = save_dir / f"{FINAL_SELECTED_MODEL}_state_dict.pt"
full_model_path = save_dir / f"{FINAL_SELECTED_MODEL}_full_model.pt"
info_path = save_dir / "selected_model_info.json"

torch.save(model.state_dict(), state_dict_path)
torch.save(model, full_model_path)

selected_info = {
    "selected_run_name": FINAL_SELECTED_MODEL,
    "model_name": best_summary["model_name"],
    "selection_reason": "Final selected model for Task 1 based on held-out test performance and overall conclusion.",
    "test_metrics": best_summary.get("test_metrics"),
    "test_session_metrics": best_summary.get("test_session_metrics"),
    "state_dict_path": str(state_dict_path),
    "full_model_path": str(full_model_path),
    "source_checkpoint_path": str(checkpoint_path),
}

with open(info_path, "w", encoding="utf-8") as f:
    json.dump(selected_info, f, indent=2)

print("Final selected model    :", FINAL_SELECTED_MODEL)
print("Saved state_dict to     :", state_dict_path)
print("Saved full model to     :", full_model_path)
print("Saved model info to     :", info_path)


Final selected model    : residual_eeg_cnn
Saved state_dict to     : /content/drive/MyDrive/Colab Notebooks/CSCI 5527/Project/code/sleep_deprivation_classification/experiments_cls2/final_selected_model/residual_eeg_cnn_state_dict.pt
Saved full model to     : /content/drive/MyDrive/Colab Notebooks/CSCI 5527/Project/code/sleep_deprivation_classification/experiments_cls2/final_selected_model/residual_eeg_cnn_full_model.pt
Saved model info to     : /content/drive/MyDrive/Colab Notebooks/CSCI 5527/Project/code/sleep_deprivation_classification/experiments_cls2/final_selected_model/selected_model_info.json


In [54]:
# This helper reloads saved results after the runs finish.
# It only shows the current focused comparison runs so older experiments do not clutter the table.
# The test columns stay empty until you run the final test cell.

def load_saved_results(experiment_root=EXPERIMENT_ROOT, allowed_run_names=None):
    rows = []
    for summary_path in sorted(experiment_root.glob("*/summary.json")):
        with open(summary_path, "r", encoding="utf-8") as f:
            summary = json.load(f)
        run_name = summary.get("run_name", summary["model_name"])
        if allowed_run_names is not None and run_name not in allowed_run_names:
            continue
        test_metrics = summary.get("test_metrics") or {}
        test_session_metrics = summary.get("test_session_metrics") or {}
        rows.append({
            "run_name": run_name,
            "model_name": summary["model_name"],
            "best_epoch": summary["best_epoch"],
            "val_accuracy": summary["val_metrics"]["accuracy"],
            "val_f1": summary["val_metrics"]["f1"],
            "val_bal_acc": summary["val_metrics"]["balanced_accuracy"],
            "val_auroc": summary["val_metrics"].get("auroc", float("nan")),
            "val_session_f1": summary["val_session_metrics"]["f1"],
            "val_session_bal_acc": summary["val_session_metrics"]["balanced_accuracy"],
            "test_evaluated": summary.get("test_evaluated", False),
            "test_accuracy": test_metrics.get("accuracy", float("nan")),
            "test_f1": test_metrics.get("f1", float("nan")),
            "test_auroc": test_metrics.get("auroc", float("nan")),
            "test_session_f1": test_session_metrics.get("f1", float("nan")),
            "elapsed_minutes": summary["elapsed_minutes"],
        })
    return pd.DataFrame(rows).sort_values(by="val_session_f1", ascending=False) if rows else pd.DataFrame()


_current_run_names = set(globals().get("MODELS_TO_RUN", [])) | set(globals().get("CV_MODELS_TO_RUN", []))
saved_results_df = load_saved_results(
    allowed_run_names=_current_run_names if len(_current_run_names) > 0 else None
)
saved_results_df


,run_name,model_name,best_epoch,val_accuracy,val_f1,val_bal_acc,val_auroc,val_session_f1,val_session_bal_acc,test_evaluated,test_accuracy,test_f1,test_auroc,test_session_f1,elapsed_minutes
1,two_branch_eeg_cnn,two_branch_eeg_cnn,11,0.770465,0.753873,0.768991,0.812953,0.900000,0.904545,True,0.607143,0.526990,0.668960,0.631579,2.042535
0,residual_eeg_cnn,residual_eeg_cnn,2,0.711075,0.717425,0.713256,0.747541,0.761905,0.763636,True,0.648176,0.616404,0.715865,0.700000,0.903677


# Includes:

- subject-wise cross-validation support
- mild temporal jitter and random time crop support for EEG CNNs
- a two-branch EEG CNN
- session-level aggregation of epoch predictions

The main models in the default run list are:
- residual EEG CNN
- two-branch EEG CNN

The notebook is now focused on the task 1 classifier from the project proposal.
Standard training compares the strongest current EEG models with train and validation only.
The held out test set is reserved for the separate final test cell.
Cross-validation is reserved for the best current model family and runs only if `RUN_CV_EXPERIMENTS = True`.

residual_eeg_cnn:

test_accuracy = 0.6482

test_f1 = 0.6164

test_bal_acc = 0.6479

test_auroc = 0.7159

test_session_f1 = 0.7000

test_session_bal_acc = 0.7273